# Community Detection Analysis

This notebook analyzes the results from the Leiden algorithm and Spectral Clustering (4 and 12 clusters).
It looks at the data from two perspectives:
1. **Community Perspective**: What percentage of a given community is made up of Januarys, Februarys, etc.?
2. **Month Perspective**: For a given month (e.g., all Januarys), what percentage of them fall into Community 1, Community 2, etc.?

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

In [ ]:
# --- Configuration & Paths ---
SCRIPT_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, '..', '..', '..'))
RESULTS_DIR = os.path.join(SCRIPT_DIR, 'results')
PLOTS_DIR = os.path.join(SCRIPT_DIR, 'plots')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Input data paths
FILES = {
    "leiden": os.path.join(PROJECT_ROOT, "networks", "community_detection", "leiden", "results", "leiden_communities.json"),
    "spectral_4": os.path.join(PROJECT_ROOT, "networks", "community_detection", "spectral_clustering", "results", "spectral_communities_4.json"),
    "spectral_12": os.path.join(PROJECT_ROOT, "networks", "community_detection", "spectral_clustering", "results", "spectral_communities_12.json")
}

MONTH_NUMS = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

In [ ]:
# --- Analysis Logic ---
def analyze_graph_communities(graph_data):
    communities = graph_data.get("communities", {})
    
    # Dictionaries to hold raw counts
    comm_counts = defaultdict(lambda: defaultdict(int)) # comm -> month -> count
    month_counts = defaultdict(lambda: defaultdict(int)) # month -> comm -> count
    
    comm_totals = defaultdict(int)
    month_totals = defaultdict(int)
    
    # Aggregate counts
    for comm, nodes in communities.items():
        for node in nodes:
            try:
                month = node.split('-')[1]
                comm_counts[comm][month] += 1
                month_counts[month][comm] += 1
                comm_totals[comm] += 1
                month_totals[month] += 1
            except IndexError:
                continue
    
    # 1. Community Perspective (rows = communities, sum = 100%)
    comm_perspective = {}
    for comm in communities.keys():
        comm_perspective[comm] = {
            m: (comm_counts[comm][m] / comm_totals[comm] * 100) if comm_totals[comm] > 0 else 0.0
            for m in MONTH_NUMS
        }
        
    # 2. Month Perspective (rows = months, sum = 100%)
    month_perspective = {}
    for month in MONTH_NUMS:
        month_perspective[month] = {
            c: (month_counts[month][c] / month_totals[month] * 100) if month_totals[month] > 0 else 0.0
            for c in communities.keys()
        }
        
    return {
        "community_perspective": comm_perspective,
        "month_perspective": month_perspective
    }

In [ ]:
# --- Run Analysis & Save to JSON ---
all_analysis = {}

for method, path in FILES.items():
    if not os.path.exists(path):
        print(f"File not found, skipping: {path}")
        continue
    
    print(f"Processing {method}...")
    with open(path, 'r') as f:
        data = json.load(f)
        
    all_analysis[method] = {}
    for graph_name, graph_data in data.items():
        all_analysis[method][graph_name] = analyze_graph_communities(graph_data)

output_json = os.path.join(RESULTS_DIR, "community_analysis_results.json")
with open(output_json, 'w') as f:
    json.dump(all_analysis, f, indent=4)
    
print(f"\nAnalysis complete. Data saved to {output_json}")

In [ ]:
# --- Visualization Logic ---
def visualize_results(analysis_data):
    for method, graphs in analysis_data.items():
        print(f"Generating plots for {method}...")
        for graph_name, results in graphs.items():
            
            # --- 1. Community Perspective Plot ---
            comm_data = results["community_perspective"]
            # Reformat for Pandas (Rows: Communities, Cols: Months)
            df_comm = pd.DataFrame(comm_data).T
            # Rename columns from "01" to "Jan"
            df_comm.columns = [MONTH_LABELS[MONTH_NUMS.index(c)] for c in df_comm.columns]
            
            # Sort communities naturally if possible
            try:
                df_comm.index = pd.CategoricalIndex(df_comm.index, 
                    categories=sorted(df_comm.index, key=lambda x: int(x.split('_')[1])), 
                    ordered=True)
                df_comm = df_comm.sort_index()
            except:
                pass # Fallback if naming convention changes
                
            plt.figure(figsize=(12, max(4, len(df_comm) * 0.5)))
            sns.heatmap(df_comm, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': '% of Community'}, vmin=0, vmax=100)
            plt.title(f"[{method.upper()}] {graph_name}\nCommunity Perspective (What months make up this community?)", fontsize=14)
            plt.ylabel("Community", fontsize=12)
            plt.xlabel("Month", fontsize=12)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_DIR, f"{method}_{graph_name}_comm_perspective.png"), dpi=300)
            plt.close()
            
            # --- 2. Month Perspective Plot ---
            month_data = results["month_perspective"]
            # Reformat for Pandas (Rows: Months, Cols: Communities)
            df_month = pd.DataFrame(month_data).T
            # Rename indices from "01" to "Jan"
            df_month.index = [MONTH_LABELS[MONTH_NUMS.index(i)] for i in df_month.index]
            
            # Sort columns naturally if possible
            try:
                cols = sorted(df_month.columns, key=lambda x: int(x.split('_')[1]))
                df_month = df_month[cols]
            except:
                pass

            plt.figure(figsize=(max(8, len(df_month.columns) * 0.8), 8))
            sns.heatmap(df_month, annot=True, fmt=".1f", cmap="Oranges", cbar_kws={'label': '% of Month'}, vmin=0, vmax=100)
            plt.title(f"[{method.upper()}] {graph_name}\nMonth Perspective (Where do these months belong?)", fontsize=14)
            plt.ylabel("Month", fontsize=12)
            plt.xlabel("Community", fontsize=12)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_DIR, f"{method}_{graph_name}_month_perspective.png"), dpi=300)
            plt.close()

In [ ]:
# Execute visualization
visualize_results(all_analysis)
print("All plots have been generated and saved to the plots/ folder.")

### Interpretation Guide
* **Blue Heatmaps (Community Perspective)**: Read *horizontally*. If you look at row "Community_1", the numbers add up to 100%. A bright blue square in the "Jan" column means a huge chunk of Community 1 consists of January nodes.
* **Orange Heatmaps (Month Perspective)**: Read *horizontally*. If you look at the row "Jan", the numbers add up to 100%. A bright orange square under "Community_2" means almost all Januarys in your dataset were grouped into Community 2.